In [1]:
import re
import itertools
from dataclasses import dataclass
from typing import List, Dict, Tuple

import pandas as pd
import torch
from transformers import pipeline

/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
texts = [
    "OpenAI was founded in 2015 by Sam Altman in San Francisco. The company later moved its headquarters to other offices.",

    "Google was founded in 1998 by Larry Page and Sergey Brin at Stanford University in California. Sundar Pichai became CEO in 2015.",

    "The University of Oxford was established in 1096 in Oxford, England. It is one of the oldest universities in Europe.",

    "Tesla was founded in 2003 in California by Martin Eberhard and Marc Tarpenning. Elon Musk joined the company later.",

    "Microsoft was founded in 1975 by Bill Gates and Paul Allen in Albuquerque. The company is based in Redmond today.",

    "Marie Curie was born in 1867 in Warsaw, Poland. She worked at the University of Paris and won a Nobel Prize.",

    "Amazon was founded in 1994 by Jeff Bezos in Seattle. The company expanded globally after 2000.",

    "Apple was founded in 1976 by Steve Jobs and Steve Wozniak in Cupertino. Tim Cook became CEO in 2011."
]

In [3]:
ner = pipeline(
    task="ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device='mps'
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps


In [4]:
sentence_split_re = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text):
    sents = [s.strip() for s in sentence_split_re.split(text) if s.strip()]
    return sents

split_sentences(texts[0])

['OpenAI was founded in 2015 by Sam Altman in San Francisco.',
 'The company later moved its headquarters to other offices.']

In [5]:
year_re = re.compile(r"\b(1[0-9]{3}|20[0-2][0-9])\b")

def extract_dates(text):
    out = []
    for m in year_re.finditer(text):
        out.append({
            "text": m.group(0),
            "type": "DATE",
            "start": m.start(),
            "end": m.end(),
            "score": 1.0,
            "source": "regex"
        })
    return out


label_map = {
    "PER": "PERSON",
    "ORG": "ORG",
    "LOC": "LOCATION",
    "MISC": "MISC"
}

def ner_entities(text):
    ents = []
    for e in ner(text):
        t = label_map.get(e["entity_group"], e["entity_group"])
        ents.append({
            "text": e["word"],
            "type": t,
            "start": int(e["start"]),
            "end": int(e["end"]),
            "score": float(e["score"]),
            "source": "model"
        })

    ents.extend(extract_dates(text))

    ents = sorted(ents, key=lambda x: (x["start"], x["end"]))
    return ents

In [6]:
def show_entities(text: str, ents: List[Dict]):
    df = pd.DataFrame(ents)[["text","type","start","end","score","source"]]
    display(df)

ents0 = ner_entities(texts[0])
show_entities(texts[0], ents0)

,text,type,start,end,score,source
0,OpenAI,ORG,0,6,0.987911,model
1,2015,DATE,22,26,1.000000,regex
2,Sam Altman,PERSON,30,40,0.993201,model
3,San Francisco,LOCATION,44,57,0.999368,model


In [7]:
def entities_in_sentence(sent, offset, all_ents):
    s0, s1 = offset, offset + len(sent)
    return [e for e in all_ents if e["start"] >= s0 and e["end"] <= s1]

def sentence_offsets(text):
    sents = []
    start = 0
    for m in sentence_split_re.finditer(text):
        end = m.start()
        sent = text[start:end].strip()
        if sent:
            true_start = text.find(sent, start, end)
            sents.append((sent, true_start))
        start = m.end()

    last = text[start:].strip()
    if last:
        true_start = text.find(last, start)
        sents.append((last, true_start))
    return sents

def entity_pairs_in_sentence(ents):
    return list(itertools.combinations(ents, 2))

def count_pairs_for_text(text):
    all_ents = ner_entities(text)
    sent_info = sentence_offsets(text)
    total_pairs = 0
    per_sent = []
    for sent, off in sent_info:
        ents = entities_in_sentence(sent, off, all_ents)
        pairs = entity_pairs_in_sentence(ents)
        per_sent.append((sent, len(ents), len(pairs)))
        total_pairs += len(pairs)
    return all_ents, per_sent, total_pairs

all_ents, per_sent, total_pairs = count_pairs_for_text(texts[0])
per_sent, total_pairs

([('OpenAI was founded in 2015 by Sam Altman in San Francisco.', 4, 6),
  ('The company later moved its headquarters to other offices.', 0, 0)],
 6)

In [8]:
def classify_relation(sent, e1, e2):
    s = sent.lower()
    t1, t2 = e1["type"], e2["type"]

    pair = {t1, t2}

    founded_kw = any(k in s for k in ["was founded", "founded in", "founded by", "established", "was established"]) 
    by_kw = " by " in s or "founded by" in s
    in_kw = " in " in s
    ceo_kw = " ceo" in s or "chief executive" in s
    works_kw = any(k in s for k in ["worked at", "works at", "joined", "employed", "at the"]) 

    if pair == {"ORG", "PERSON"} and founded_kw and by_kw:
        return "FOUNDED_BY"

    if pair == {"ORG", "DATE"} and founded_kw:
        return "FOUNDED_IN_YEAR"

    if pair == {"ORG", "LOCATION"} and ("based in" in s or (founded_kw and in_kw) or "headquarters" in s):
        return "LOCATED_IN"

    if pair == {"ORG", "PERSON"} and ceo_kw:
        return "CEO_OF"

    if pair == {"ORG", "PERSON"} and works_kw:
        return "WORKS_AT"

    return "NO_RELATION"

In [10]:
def run_ie(text: str):
    ents = ner_entities(text)
    sent_info = sentence_offsets(text)

    rel_rows = []
    pair_stats = []

    for sent, off in sent_info:
        sent_ents = entities_in_sentence(sent, off, ents)
        pairs = entity_pairs_in_sentence(sent_ents)
        pair_stats.append({
            "sentence": sent,
            "num_entities": len(sent_ents),
            "num_pairs": len(pairs)
        })
        for e1, e2 in pairs:
            rel = classify_relation(sent, e1, e2)
            rel_rows.append({"sentence": sent,
                             "e1": e1["text"], "e1_type": e1["type"],
                             "e2": e2["text"], "e2_type": e2["type"],
                             "relation": rel})

    return ents, pd.DataFrame(pair_stats), pd.DataFrame(rel_rows)

results = []
for i, text in enumerate(texts):
    ents, pair_stats_df, rel_df = run_ie(text)
    results.append((ents, pair_stats_df, rel_df))
    
    print(f"\n=== TEXT {i} ===")
    print(text)
    print("\nEntities:")
    display(pd.DataFrame(ents)[["text","type","start","end","source"]])
    print("\nPairs per sentence:")
    display(pair_stats_df)
    print("\nRelations (candidates):")
    display(rel_df)


=== TEXT 0 ===
OpenAI was founded in 2015 by Sam Altman in San Francisco. The company later moved its headquarters to other offices.

Entities:


,text,type,start,end,source
0,OpenAI,ORG,0,6,model
1,2015,DATE,22,26,regex
2,Sam Altman,PERSON,30,40,model
3,San Francisco,LOCATION,44,57,model



Pairs per sentence:


,sentence,num_entities,num_pairs
0,OpenAI was founded in 2015 by Sam Altman in Sa...,4,6
1,The company later moved its headquarters to ot...,0,0



Relations (candidates):


...

,text,type,start,end,source
0,Apple,ORG,0,5,model
1,1976,DATE,21,25,regex
2,Steve Jobs,PERSON,29,39,model
3,Steve Wozniak,PERSON,44,57,model
4,Cupertino,LOCATION,61,70,model
5,Tim Cook,PERSON,72,80,model
6,2011,DATE,95,99,regex



Pairs per sentence:


,sentence,num_entities,num_pairs
0,Apple was founded in 1976 by Steve Jobs and St...,5,10
1,Tim Cook became CEO in 2011.,2,1



Relations (candidates):


,sentence,e1,e1_type,e2,e2_type,relation
0,Apple was founded in 1976 by Steve Jobs and St...,Apple,ORG,1976,DATE,FOUNDED_IN_YEAR
1,Apple was founded in 1976 by Steve Jobs and St...,Apple,ORG,Steve Jobs,PERSON,FOUNDED_BY
2,Apple was founded in 1976 by Steve Jobs and St...,Apple,ORG,Steve Wozniak,PERSON,FOUNDED_BY
3,Apple was founded in 1976 by Steve Jobs and St...,Apple,ORG,Cupertino,LOCATION,LOCATED_IN
4,Apple was founded in 1976 by Steve Jobs and St...,1976,DATE,Steve Jobs,PERSON,NO_RELATION
5,Apple was founded in 1976 by Steve Jobs and St...,1976,DATE,Steve Wozniak,PERSON,NO_RELATION
6,Apple was founded in 1976 by Steve Jobs and St...,1976,DATE,Cupertino,LOCATION,NO_RELATION
7,Apple was founded in 1976 by Steve Jobs and St...,Steve Jobs,PERSON,Steve Wozniak,PERSON,NO_RELATION
8,Apple was founded in 1976 by Steve Jobs and St...,Steve Jobs,PERSON,Cupertino,LOCATION,NO_RELATION
9,Apple was founded in 1976 by Steve Jobs and St...,Steve Wozniak,PERSON,Cupertino,LOCATION,NO_RELATION


In [11]:
chosen = 1
ents, pair_stats_df, rel_df = results[chosen]

print("TEXT:")
print(texts[chosen])

print("\nEntities:")
display(pd.DataFrame(ents)[["text","type","source"]])

print("\nPairs per sentence (count):")
display(pair_stats_df)
print("Total pairs:", int(pair_stats_df["num_pairs"].sum()))

print("\nAll relation candidates:")
display(rel_df.sort_values(["relation","e1","e2"]))

TEXT:
Google was founded in 1998 by Larry Page and Sergey Brin at Stanford University in California. Sundar Pichai became CEO in 2015.

Entities:


,text,type,source
0,Google,ORG,model
1,1998,DATE,regex
2,Larry Page,PERSON,model
3,Sergey Brin,PERSON,model
4,Stanford University,ORG,model
5,California,LOCATION,model
6,Sundar Pichai,PERSON,model
7,2015,DATE,regex



Pairs per sentence (count):


,sentence,num_entities,num_pairs
0,Google was founded in 1998 by Larry Page and S...,6,15
1,Sundar Pichai became CEO in 2015.,2,1


Total pairs: 16

All relation candidates:


,sentence,e1,e1_type,e2,e2_type,relation
1,Google was founded in 1998 by Larry Page and S...,Google,ORG,Larry Page,PERSON,FOUNDED_BY
2,Google was founded in 1998 by Larry Page and S...,Google,ORG,Sergey Brin,PERSON,FOUNDED_BY
10,Google was founded in 1998 by Larry Page and S...,Larry Page,PERSON,Stanford University,ORG,FOUNDED_BY
12,Google was founded in 1998 by Larry Page and S...,Sergey Brin,PERSON,Stanford University,ORG,FOUNDED_BY
7,Google was founded in 1998 by Larry Page and S...,1998,DATE,Stanford University,ORG,FOUNDED_IN_YEAR
0,Google was founded in 1998 by Larry Page and S...,Google,ORG,1998,DATE,FOUNDED_IN_YEAR
4,Google was founded in 1998 by Larry Page and S...,Google,ORG,California,LOCATION,LOCATED_IN
14,Google was founded in 1998 by Larry Page and S...,Stanford University,ORG,California,LOCATION,LOCATED_IN
8,Google was founded in 1998 by Larry Page and S...,1998,DATE,California,LOCATION,NO_RELATION
5,Google was founded in 1998 by Larry Page and S...,1998,DATE,Larry Page,PERSON,NO_RELATION


### Wnioski

Zdanie: "Google was founded in 1998 by Larry Page and Sergey Brin at Stanford University in California. Sundar Pichai became CEO in 2015."

**Poprawne relacje**:
- (Google, Larry Page) → FOUNDED_BY
- (Google, Sergey Brin) → FOUNDED_BY
- (Google, 1998) → FOUNDED_IN_YEAR

2) **Błędne/wątpliwe**:
- (Google, California) → LOCATED_IN (zależy od interpretacji, to raczej ma być FOUNDED_IN)
- (Larry Page, Stanford University) → FOUNDED_BY: Błędne, przyczyna: prosty `if` wykrył słowa „founded … by” oraz parę (PERSON, ORG)
- (Sergey Brin, Stanford University) → FOUNDED_BY: Błędne, przyczyna taka sama
- (1998, Stanford University) → FOUNDED_IN_YEAR: Błędne, zadziałało tak samo jak dla `(Google, 1998) → FOUNDED_IN_YEAR`